# Controlling MODFLOW 6 with the API — C. Adjusting recharge with a callback

This is part C of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle and the callback mechanism.

Modify model inputs while the simulation is running. A `modflowapi` callback intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation on the fly without rewriting the input files. Then confirm the change directly from the model's own budget output.

Run the imports and library-location cells below first.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline
import os
import pathlib as pl
import platform

import flopy
import matplotlib.pyplot as plt
import numpy as np
from modflowapi import Callbacks, run_simulation

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. Find the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the executable inside the active environment (here the project's pixi environment), and confirm that both exist.

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the pixi environment"

# the shared library and executable file extensions differ by platform
system = platform.platform().lower()
if "linux" in system:
    lib_ext, exe_ext = ".so", ""
elif "darwin" in system or "macos" in system:
    lib_ext, exe_ext = ".dylib", ""
else:
    lib_ext, exe_ext = ".dll", ".exe"

# get_mf6.py installs mf6 into different subdirectories depending on platform
# (bin on Windows; bin plus lib on Unix). Check the usual locations first, then
# fall back to a recursive search: meson installs the Linux .so under a
# multiarch subdir such as lib/x86_64-linux-gnu that a fixed list would miss.
search_dirs = ["bin", "lib", "Scripts", "Library/bin"]


def find_in_env(filename):
    for d in search_dirs:
        candidate = env_path / d / filename
        if candidate.is_file():
            return candidate
    matches = sorted(env_path.rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} under {env_path} "
        f"(looked in {', '.join(search_dirs)} and recursively)"
    )


lib_name = find_in_env(f"libmf6{lib_ext}")
mf6_exe = find_in_env(f"mf6{exe_ext}")

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Increase precipitation with a callback

Modify model inputs while the simulation is running. The `callback_function` below is called by `modflowapi` at key points in the solution (for example `Callbacks.initialize`, `Callbacks.stress_period_start`, and `Callbacks.iteration_start`); here it intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation on the fly without rewriting the input files.

The callback is the most convenient way to work with the standard stress packages. `modflowapi` exposes the running model as familiar Python objects - the simulation (`sim`), each model (`sim.sv`), and its packages (`ml.rch_0`) - so you can reach a package by name and read or change its `stress_period_data` directly with numpy-style syntax (`spd["recharge"] *= 1.1`). The edits take effect in the running model immediately, and you never have to know the internal MODFLOW variable names, memory addresses, or array layout.

For data the package interface does not expose - internal solver variables or advanced-package terms, for example - you fall back to the lower-level `get_value` / `get_value_ptr` memory access introduced at the top of the notebook. The streamflow-augmentation example below uses it to read the simulated SFR flow and set the prediction-well rate.

In [ ]:
ws = pl.Path(f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}")
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
def callback_function(sim, callback_step):
    """
    A demonstration function that dynamically adjusts recharge in a
    MODFLOW 6 model through the MODFLOW-API

    Parameters
    ----------
    sim : modflowapi.ApiSimulation
        A simulation object for the solution group that is
        currently being solved
    step : enumeration
        modflowapi.Callbacks enumeration object that indicates
        the part of the solution modflow is currently in.
    """
    ml = sim.sv
    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        rcha = ml.rch_0
        spd = rcha.stress_period_data
        print(f"updating recharge: stress_period={ml.kper + 1}")
        print(spd)
        spd["recharge"] *= 1.1

    if callback_step == Callbacks.timestep_start:
        pass

    if callback_step == Callbacks.iteration_start:
        # we can implement complex solutions to boundary conditions here!
        pass

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Confirm the callback changed the results

Running a callback is only convincing if you can see its effect. The cell below reads the **total global recharge** from the volumetric budget (`BUDGETCSV`) that the run above just wrote, and compares it with the original recharge.

There is no need to run the model a second time to get the baseline. The callback multiplies recharge by 1.1 in *every* stress period, so the scaling is uniform and the original recharge is simply the increased recharge divided by 1.1 - both curves come from this single run. (If a callback instead applied a *non-uniform* change, you would record the before/after values inside the callback itself, or compare against a separately saved baseline budget.) The two curves stay a constant 10% apart, which is the visual confirmation that the callback took effect.

In [ ]:
# read the total global recharge from the budget CSV the callback run just wrote.
# the callback scaled recharge by 1.1 uniformly, so the original recharge is just
# the increased recharge / 1.1 - no second model run is needed.
gwf = sim.get_model(name)
budget = gwf.output.budgetcsv().data

period = np.arange(1, len(budget) + 1)
rch_increased = budget["RCH(RCH_0)_IN"]
rch_baseline = rch_increased / 1.1

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(
        period,
        rch_baseline,
        marker="o",
        ms=4,
        lw=1.5,
        color="0.4",
        drawstyle="steps-mid",
        label="Original recharge",
    )
    ax.plot(
        period,
        rch_increased,
        marker="o",
        ms=4,
        lw=1.5,
        color="blue",
        drawstyle="steps-mid",
        label="Increased recharge (callback, +10%)",
    )
    flopy.plot.styles.heading(ax=ax, heading="Total global recharge")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge rate, ft$^3$/d")
    flopy.plot.styles.remove_edge_ticks(ax=ax)
    ax.legend(loc="best", frameon=False)
    fig.savefig(fig_path / "sv-recharge-callback.png", dpi=300, transparent=False)